# Convergence witness — counter vs AdamW vs bf16 (one Colab session)

Three arms, IDENTICAL FineWeb-Edu tokens, same GLM architecture (RMSNorm+GQA+RoPE+QK-norm), equal ACTIVE compute:

| arm | weights | optimizer | question it answers |
|---|---|---|---|
| **A counter-fp32** | ternary+6-bit counter | in-state | baseline of the method |
| **B counter-bf16** | same, bf16 expert GEMMs | in-state | does bf16 (×1.5-2 speed) hold quality over a LONG horizon? |
| **C dense-AdamW** | fp32 dense | AdamW | does the counter optimizer track AdamW convergence? |

~25M tokens/arm (≈20× the toy run where bf16 lagged) — a few hours total on a T4, less on L4/A100. Package embedded (no clone). Runtime → GPU → Run all. Curves saved to `curves.json` after every eval, so partial results survive a disconnect.

In [ ]:
!pip -q install tiktoken datasets 2>/dev/null
print('deps ok')

In [ ]:
import base64, io, tarfile, sys, os
PKG_B64 = "..."
os.makedirs("/content/mn", exist_ok=True)
tarfile.open(fileobj=io.BytesIO(base64.b64decode(PKG_B64)), mode="r:gz").extractall("/content/mn")
sys.path.insert(0, "/content/mn")
import memory_native; print("memory_native ready")

In [ ]:
# ---- data: tokenize ONCE, all arms consume the SAME buffer (fair + deterministic) ----
import torch, time
STEPS, B, T, EVAL_EVERY, EVAL_BATCHES = 3000, 16, 512, 250, 20
VAL_TOKENS = 300_000
TIME_CAP_PER_ARM = 75*60          # safety: cap each arm at 75 min; curves are saved as it goes

import tiktoken
enc = tiktoken.get_encoding("gpt2"); VOCAB = enc.n_vocab
need = STEPS*B*(T+1) + VAL_TOKENS
print(f"tokenizing {need/1e6:.1f}M FineWeb tokens once (few minutes)...", flush=True)
from datasets import load_dataset
ds = load_dataset("HuggingFaceFW/fineweb-edu", name="sample-10BT", split="train", streaming=True)
buf, t0 = [], time.time()
for doc in ds:
    buf.extend(enc.encode_ordinary(doc["text"])); buf.append(enc.eot_token)
    if len(buf) >= need: break
TOK = torch.tensor(buf[:need], dtype=torch.int32); del buf
VAL = TOK[:VAL_TOKENS].long()                       # held out BEFORE training tokens
TRAINTOK = TOK[VAL_TOKENS:]
print(f"tokens ready: train {TRAINTOK.numel()/1e6:.1f}M + val {VAL.numel()/1e6:.2f}M "
      f"({time.time()-t0:.0f}s)", flush=True)

In [ ]:
import json, math, time, torch, torch.nn as nn, torch.nn.functional as F
from memory_native.glm import MNGLM, GLMAttention, RMSNorm

assert torch.cuda.is_available(), "Runtime -> GPU"
dev = "cuda"
D, L_, NH, NKV, E_, TOPK = 512, 8, 8, 2, 8, 2      # d=512 -> int8 OFF (net-negative below d=768)

# dense-AdamW arm: SAME attention stack (RMSNorm+GQA+RoPE+QK-norm via memory_native.glm), dense
# SwiGLU FFN sized to the SAME ACTIVE MACs as the MoE arms (3*d*H == top_k*3*d*h -> H = top_k*h).
class DenseGLM(nn.Module):
    def __init__(self, vocab, d, nl, nh, nkv, T, h_expert):
        super().__init__()
        self.d, self.block_size = d, T
        H = TOPK * h_expert
        self.tok = nn.Embedding(vocab, d); nn.init.normal_(self.tok.weight, std=0.02)
        def ffn():
            return nn.ModuleDict(dict(g=nn.Linear(d, H, bias=False), u=nn.Linear(d, H, bias=False),
                                      dn=nn.Linear(H, d, bias=False)))
        self.blocks = nn.ModuleList(nn.ModuleDict(dict(
            n1=RMSNorm(d), attn=GLMAttention(d, nh, nkv, "dense", {}, qk_norm=True),
            n2=RMSNorm(d), ffn=ffn())) for _ in range(nl))
        self.nf = RMSNorm(d)
        self.head = nn.Linear(d, vocab, bias=False); self.head.weight = self.tok.weight
    def forward(self, idx, targets=None, loss_chunk=2048):
        x = self.tok(idx)
        for b in self.blocks:
            x = x + b["attn"](b["n1"](x))
            h = b["n2"](x); f = b["ffn"]
            x = x + f["dn"](F.silu(f["g"](h)) * f["u"](h))
        h = self.nf(x)
        if targets is None: return self.head(h), None
        hf = h.reshape(-1, self.d); tf = targets.reshape(-1); tot = hf.new_zeros(())
        for i in range(0, hf.shape[0], loss_chunk):
            tot = tot + F.cross_entropy(self.head(hf[i:i+loss_chunk]), tf[i:i+loss_chunk], reduction="sum")
        return None, tot / hf.shape[0]

def build(arm):
    torch.manual_seed(0)
    if arm == "dense_adamw":
        probe = MNGLM(VOCAB, D, 1, NH, NKV, T, n_experts=E_, top_k=TOPK, grouped=True, swiglu=True)
        return DenseGLM(VOCAB, D, L_, NH, NKV, T, probe.blocks[0].ffn.h).to(dev).train()
    dt = "bf16" if arm == "counter_bf16" else "fp32"
    return MNGLM(VOCAB, D, L_, NH, NKV, T, kind="counter_packed", n_experts=E_, top_k=TOPK,
                 qk_norm=True, grouped=True, swiglu=True, moe_dtype=dt, act_save_bits=8).to(dev).train()

def batches():
    per = B*(T+1)
    for s in range(STEPS):
        c = TRAINTOK[s*per:(s+1)*per].long().view(B, T+1).to(dev)
        yield c[:, :T].contiguous(), c[:, 1:].contiguous()

@torch.no_grad()
def evaluate(m):
    m.eval(); tot = 0.0; g = torch.Generator(device=dev).manual_seed(0); vd = VAL.to(dev)
    for _ in range(EVAL_BATCHES):
        i = torch.randint(0, vd.numel()-T-1, (B,), generator=g, device=dev)
        x = torch.stack([vd[j:j+T] for j in i]); y = torch.stack([vd[j+1:j+1+T] for j in i])
        tot += m(x, y)[1].item()
    m.train(); return tot / EVAL_BATCHES

curves = {}
for arm in ("counter_fp32", "counter_bf16", "dense_adamw"):
    m = build(arm)
    opt = torch.optim.AdamW([p for p in m.parameters() if p.requires_grad], lr=3e-4)
    pts, t0, s = [], time.time(), 0
    print(f"\n==== {arm} ====", flush=True)
    for x, y in batches():
        _, loss = m(x, y)
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); s += 1
        if s % EVAL_EVERY == 0 or s == STEPS:
            vl = evaluate(m); el = time.time() - t0
            pts.append({"step": s, "tokens": s*B*T, "val": vl, "tok_s": s*B*T/el})
            print(f"  step {s:5d}  {s*B*T/1e6:5.1f}M tok  val {vl:.4f}  {s*B*T/el:,.0f} tok/s", flush=True)
            curves[arm] = pts; json.dump(curves, open("curves.json", "w"))
        if time.time() - t0 > TIME_CAP_PER_ARM:
            print(f"  time cap hit at step {s}", flush=True); break
    del m; torch.cuda.empty_cache()

print("\n==== FINAL (loss @ equal tokens) ====")
for arm, pts in curves.items():
    print(f"  {arm:14s} val {pts[-1]['val']:.4f} @ {pts[-1]['tokens']/1e6:.1f}M tok, {pts[-1]['tok_s']:,.0f} tok/s")
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(7,4.5))
    for arm, pts in curves.items():
        plt.plot([p["tokens"]/1e6 for p in pts], [p["val"] for p in pts], marker="o", label=arm)
    plt.xlabel("tokens (M)"); plt.ylabel("val loss"); plt.title("convergence: counter vs AdamW vs bf16")
    plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.savefig("curves.png", dpi=120); plt.show()
    print("saved: curves.json, curves.png")
except Exception as e:
    print("plot skipped:", e, "| curves.json has the data")